# Lab 13: How Fast Are the Days Getting Shorter?

**MATH 170 — Introduction to Scientific Computing**
**Fall 2026**

In Lab 9 you worked out the hours of daylight in Atlanta, and `daily_change`:
the minutes of daylight gained from one day to the next. That was a derivative,
estimated with a forward difference and a step of one day.

This week you grow the family of classes for numerical differentiation from
lecture: one method that every member inherits, a new member that looks back
instead of ahead, a member built on top of another member, and a test for the
whole family. At the end, the family tells you how fast the days are getting
shorter today.

Everything comes from **§9.1–9.2**: class hierarchies and inheritance,
`super()`, and classes for numerical differentiation, together with `__call__`
(§8.3) and test functions (§8.5) from Chapter 8.

---

## Before anything else

Run this cell first.

In [ ]:
# Just run this one.
import os

print("ready -- working in", os.getcwd())
print("lab13_daylight.py is here:", os.path.exists("lab13_daylight.py"))

If the folder printed above is not your `lab13` folder, or the file says
`False`, close the notebook and open it again from inside `lab13`.

---

## What you will do

**Tasks 1–4**: three pieces of a class hierarchy and one test function. Work
each one out here in the notebook, where you can run it and see what it does.
At the end, one cell collects your work and writes it into
**`lab13_daylight.py`** for you.

That file is the one that gets graded. It travels: your laptop → your own
repository on GitHub (`git push`) → Gradescope, which reads it from there.

- **Task 1**: `values`, one method in the base class for every member of the family.
- **Task 2**: `Backward1`, a new member that looks one step back.
- **Task 3**: `Richardson`, a member built on top of `Central2`.
- **Task 4**: a test function for the whole family.

The `class` lines, the `def` lines and the docstrings are given: fill in the
body of each method, and leave those lines as they are. After you change a
class, run its cell again before its TEST cell.

**One thing from lecture matters a lot here:** a subclass keeps the base class
it was made from. If you change `Diff` and run its cell again, run the
`Backward1` and `Richardson` cells again too. The TEST cells tell you if you
forget.

The build cell copies the **classes and your test function** into the file, and
nothing else from the notebook. Everything they use must be written inside them.

**Grading is on completion.** Each of the four tasks is worth **25 points**,
earned by doing real work on it — your answer does not have to be right. The
autograder still shows you which checks passed, so use it to find what to fix.

---

## The family so far

In lecture you wrote a base class, `Diff`, that keeps the function `f` and the
step `h`, and can change the step with `set_stepsize`. Each subclass adds only
`__call__`, with its own formula, so that an object like `Forward1(f, h)`
behaves like $f'$:

$$\texttt{Forward1:}\quad f'(x) \approx \frac{f(x+h) - f(x)}{h}, \qquad\qquad
\texttt{Central2:}\quad f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}.$$

The cell for Task 1 has all three classes in it. `Forward1` and `Central2` are
exactly as you wrote them; leave them as they are.

---

## Task 1 — One method for the whole family (§9.1)

Add a method `values(xs)` to `Diff`: given a list of points `xs`, it returns a
**list** with the derivative at each of them, in the same order.

`Diff` has no `__call__` of its own, and that is fine. `values` is never used on
a plain `Diff`, only on a `Forward1`, a `Central2`, or a member you write later,
and each of those has a `__call__`. So inside `values`, `self(x)` is the
derivative at `x`, worked out with **whichever formula the object has**. Write
`values` once, in the base class, and every member of the family gets it, the
same way `Parabola` got `table` from `Line` in lecture.

**On paper first:** take $f(x) = x^2$ and a step of $h = 1$. What is
`Forward1(f, 1).values([0, 1, 2])`? And `Central2(f, 1).values([0, 1, 2])`?
Which one gives exactly $f'(x) = 2x$? (Check: Forward1 gives
$(x+1)^2 - x^2 = 2x + 1$.)

In [ ]:
# Task 1 is values, at the end of Diff. After any change in this cell, run it,
# then run the Backward1 and Richardson cells again.

class Diff:
    """Base class for numerical derivatives: keeps the function f and the step h."""

    def __init__(self, f, h=1E-5):
        self.f, self.h = f, h

    def set_stepsize(self, h):
        self.h = h

    def values(self, xs):
        """
        The derivative at every point in the list xs, as a list: one number
        for each x, in the same order. It must work for every subclass, each
        with its own formula.
        """
        # TODO: your code here
        return None


# Forward1 and Central2, as you wrote them in lecture. Leave them as they are.

class Forward1(Diff):
    def __call__(self, x):
        f, h = self.f, self.h
        return (f(x+h) - f(x))/h


class Central2(Diff):
    def __call__(self, x):
        f, h = self.f, self.h
        return (f(x+h) - f(x-h))/(2*h)

In [ ]:
# TEST -- Task 1
# The try/except here is a tool, not an exercise: it turns a crash into a hint.
def square(x):
    return x*x


def near(got, expected):
    # True when got is a list of numbers within 1e-9 of the list expected.
    try:
        return len(got) == len(expected) and all(abs(a - b) < 1e-9 for a, b in zip(got, expected))
    except TypeError:
        return False


got = 'crashed'
try:
    forward = Forward1(square, 1).values([0, 1, 2])
    central = Central2(square, 1).values([0, 1, 2, 3])
    empty = Central2(square, 1).values([])
    print("Forward1(square, 1).values([0, 1, 2])    =", forward, "   expected [1.0, 3.0, 5.0]")
    print("Central2(square, 1).values([0, 1, 2, 3]) =", central, "   expected [0.0, 2.0, 4.0, 6.0]")
    print("Central2(square, 1).values([])           =", empty, "   expected []")
    got = [forward, central, empty]
except RecursionError:
    print("values crashed with RecursionError: it calls itself, again and again.")
    print("The derivative at x is self(x), not self.values(...).")
except TypeError as e:
    print("values crashed with TypeError:", e)
    if "not iterable" in str(e):
        print("xs is a list of points: go through it with a loop or a list comprehension.")
    elif "not callable" in str(e):
        print("Call the object itself: self(x). self.f is f, and self.h is a number.")
    else:
        print("Look at the line the error points to. Is something still None?")
except NameError as e:
    print("values crashed with NameError:", e)
    print("Inside a method, the object is self: self(x), self.f, self.h.")

if got == 'crashed':
    pass
elif got[0] is None:
    print("Not implemented yet -- values still returns None.")
elif not isinstance(got[0], (list, tuple)):
    print("values must return a list: one derivative for every x in xs.")
elif near(got[0], [0, 1, 4]):
    print("Those are the values of f itself: self.f(x) is f. The derivative is self(x).")
elif near(got[1], [1, 3, 5, 7]):
    print("Central2 gave Forward1's numbers: values has a formula of its own. Let each")
    print("object work out its own derivative, self(x), so that values works for every")
    print("subclass.")
elif len(got[0]) != 3 or len(got[1]) != 4:
    print("One derivative for every x in xs: three points, three numbers.")
elif not (near(got[0], [1, 3, 5]) and near(got[1], [0, 2, 4, 6])):
    print("Wrong numbers. Is the derivative at x self(x)?")
elif len(got[2]) != 0:
    print("No points, no derivatives: values([]) should be [].")
else:
    print("PASSED")

---

## Task 2 — Looking back (§9.2)

`Forward1` looks one step ahead. `Backward1` looks one step back:

$$f'(x) \approx \frac{f(x) - f(x-h)}{h}.$$

With a step of one day, that is *today minus yesterday*.

`Backward1` is a subclass of `Diff`. Like the subclasses from lecture, it has
**only** `__call__`: the constructor, `set_stepsize` and `values` all come from
`Diff`.

**On paper first:** take $f(x) = x^3$ at $x = 1$, with a big step, $h = 2$. The
true derivative is $f'(1) = 3$.

1. What does Forward1 give? And Backward1?
2. What is the average of the two? Compare it with what Central2 gives. Is that
   a coincidence?

(Check: Forward1 gives 13.)

In [ ]:
class Backward1(Diff):
    """f'(x) from f(x) and f(x - h): one step back."""

    def __call__(self, x):
        """The backward-difference approximation of f'(x), with step self.h."""
        # TODO: your code here
        return None

In [ ]:
# TEST -- Task 2
# The try/except here is a tool, not an exercise: it turns a crash into a hint.
from math import sin, cos


def cube(x):
    return x**3


result = 'crashed'
if not issubclass(Backward1, Diff):
    result = 'old'
    print("Backward1 was made from an older Diff: the Task 1 cell was run again after")
    print("the Backward1 cell. Run the Backward1 cell again, then this one.")
else:
    try:
        back = Backward1(cube, 2)(1)
        print("Backward1(cube, 2)(1) =", back, "   expected 1.0")
        if back is None:
            result = None
        else:
            ahead, middle = Forward1(cube, 2)(1), Central2(cube, 2)(1)
            print("Forward1(cube, 2)(1)  =", ahead)
            print("the average of the two =", (ahead + back)/2, "   Central2(cube, 2)(1) =", middle)
            b = Backward1(cube, 2)
            before = b(2)
            b.set_stepsize(1)
            after = b(2)
            print("Backward1(cube, 2)(2)  =", before, "   expected 4.0")
            print("after set_stepsize(1)  =", after, "   expected 7.0")
            small = Backward1(sin)(1)
            print("Backward1(sin)(1)      =", small, "   cos(1) =", cos(1))
            result = [back, before, after, small]
    except ZeroDivisionError as e:
        print("Backward1 crashed with ZeroDivisionError:", e)
        print("Divide by the step h, which is never 0 here.")
    except NameError as e:
        print("Backward1 crashed with NameError:", e)
        print("f and h are attributes: self.f and self.h. Or copy them first, as Forward1")
        print("does: f, h = self.f, self.h")
    except AttributeError as e:
        print("Backward1 crashed with AttributeError:", e)
        print("Did Backward1 get an __init__ of its own? It does not need one: the")
        print("constructor comes from Diff.")
    except TypeError as e:
        print("Backward1 crashed with TypeError:", e)
        print("Look at the line the error points to. Did a def line lose its self?")

if result in ('crashed', 'old'):
    pass
elif result is None:
    print("Not implemented yet -- __call__ still returns None.")
elif abs(result[0] - 13) < 1e-9:
    print("That is Forward1's answer. Backward1 looks back: it uses f(x) and f(x - h).")
elif abs(result[0] + 1) < 1e-9:
    print("Right size, wrong sign: f(x) comes first, then f(x - h).")
elif abs(result[0] - 2) < 1e-9:
    print("Divide by the step h.")
elif abs(result[0] - 1) > 1e-9:
    print("Work it out by hand: f(1) = 1, f(-1) = -1, and the step is 2.")
elif abs(result[1] - 4) > 1e-9 or abs(result[2] - 7) > 1e-9:
    print("set_stepsize must change the answer: use the object's own step, self.h,")
    print("not a fixed number.")
elif abs(result[3] - cos(1)) > 1e-4:
    print("With the default step, 1E-5, Backward1(sin)(1) should be very close to cos(1).")
else:
    print("PASSED")

---

## Task 3 — A subclass of a subclass (§9.1, §9.2)

In lecture you saw Central2's error shrink about 4 times every time the step
is halved. **Richardson extrapolation** uses that to do better. Write $D(h)$ for
Central2's answer with step $h$. Then

$$f'(x) \approx \frac{4\,D(h/2) - D(h)}{3}$$

cancels most of the error that is left.

`Richardson` is a subclass of **`Central2`**, not of `Diff`: it does what
Central2 does, twice, and combines the two answers. The hierarchy now has three
levels: `Diff` → `Central2` → `Richardson`. Its `__call__` must

1. remember the step the object has, `h`;
2. get $D(h)$ from Central2's `__call__`, which inside `Richardson` is
   `super().__call__(x)`;
3. halve the step with `set_stepsize`, a method `Richardson` inherits from two
   levels up, and get $D(h/2)$ the same way;
4. **put the step back** to `h`, so that the object is the same after the call
   as it was before;
5. return the combination.

Why `super().__call__(x)` and not `self(x)`? Inside `Richardson`, `self(x)` is
Richardson's own `__call__`, the method you are writing, so it would call itself
again, and again, forever.

**On paper first:**

1. Take $f(x) = x^3$ at $x = 1$ with $h = 2$ again. Central2 gave $D(2) = 7$.
   What is $D(1)$? What does Richardson give? Compare with $f'(1) = 3$.
2. In lecture, `Central4` was
   $$\frac{4}{3}\,\frac{f(x+h) - f(x-h)}{2h} - \frac{1}{3}\,\frac{f(x+2h) - f(x-2h)}{4h}.$$
   Write it with $D$. Then show, with no numbers at all, that
   `Richardson(f, h)` gives exactly what `Central4(f, h/2)` gives.

In [ ]:
class Richardson(Central2):
    """Two Central2 answers, with steps h and h/2, combined into a better one."""

    def __call__(self, x):
        """
        (4*D(h/2) - D(h))/3, where D(step) is Central2's answer with that step.
        Get each D with super().__call__(x), change the step with set_stepsize,
        and leave the step as it was before returning.
        """
        # TODO: your code here
        return None

In [ ]:
# TEST -- Task 3
# The try/except here is a tool, not an exercise: it turns a crash into a hint.
from math import sin, cos


def cube(x):
    return x**3


class Central4(Diff):
    # Central4 from lecture, to compare with.
    def __call__(self, x):
        f, h = self.f, self.h
        return (4./3)*(f(x+h) - f(x-h))/(2*h) - \
               (1./3)*(f(x+2*h) - f(x-2*h))/(4*h)


def near(a, b):
    return abs(a - b) < 1e-9


result = 'crashed'
if not issubclass(Richardson, Central2):
    result = 'old'
    print("Richardson was made from an older Central2: the Task 1 cell was run again")
    print("after the Richardson cell. Run the Richardson cell again, then this one.")
else:
    try:
        r = Richardson(cube, 2)
        first = r(1)
        step = r.h
        again = r(1)
        print("Richardson(cube, 2)(1)  =", first, "   expected 3.0 -- exactly f'(1)")
        print("its step after the call =", step, "   expected 2, as it was")
        print("called a second time    =", again, "   expected 3.0 again")
        mine = Richardson(sin, 0.5)(1)
        lecture = Central4(sin, 0.25)(1)
        print("Richardson(sin, 0.5)(1) =", mine)
        print("Central4(sin, 0.25)(1)  =", lecture, "   the same number?")
        print("isinstance(r, Central2) =", isinstance(r, Central2),
              "   isinstance(r, Diff) =", isinstance(r, Diff))
        result = [first, step, again, mine, lecture]
    except RecursionError:
        print("Richardson crashed with RecursionError: its __call__ calls itself, again and again.")
        print("self(x) is Richardson's own __call__. Central2's is super().__call__(x).")
    except TypeError as e:
        print("Richardson crashed with TypeError:", e)
        if "NoneType" in str(e):
            print("Something is None. Does every step keep its answer in a variable?")
        else:
            print("Look at the line the error points to.")
    except (AttributeError, NameError) as e:
        print("Richardson crashed with", type(e).__name__ + ":", e)
        print("The step is self.h, and set_stepsize is a method: self.set_stepsize(...).")
    except ZeroDivisionError as e:
        print("Richardson crashed with ZeroDivisionError:", e)
        print("Is the step 0 somewhere? Halve it with h/2, and put h back at the end.")

if result in ('crashed', 'old'):
    pass
elif result[0] is None:
    print("Not implemented yet -- __call__ still returns None.")
elif near(result[0], 7):
    print("That is Central2's own answer, D(h): (4*D - D)/3 is D. Both calls to")
    print("super().__call__ used the same step. Halve it with set_stepsize between them.")
elif near(result[0], 8):
    print("The 4 goes with D(h/2), the answer with the SMALLER step: (4*D(h/2) - D(h))/3.")
elif not near(result[1], 2):
    print("After the call the object's step is", result[1], "instead of 2. Put it back with")
    print("set_stepsize before you return.")
elif not near(result[2], result[0]):
    print("The same call gave a different answer the second time: the first call changed")
    print("the object. Put the step back with set_stepsize before you return.")
elif not near(result[0], 3):
    print("Work it out by hand: D(2) = 7 and D(1) = 4. The second step is h/2.")
elif abs(result[3] - result[4]) > 1e-12:
    print("Right for cube, but not the same as Central4 with half the step. Is the second")
    print("step h/2?")
else:
    print("PASSED")
    print()
    print("Halving the step, for sin at x = 1:")
    print("      h     Central2 error   Richardson error")
    for h in [1, 0.5, 0.25, 0.125]:
        e2 = abs(Central2(sin, h)(1) - cos(1))
        er = abs(Richardson(sin, h)(1) - cos(1))
        print(f"{h:7.3f}   {e2:14.2e}   {er:16.2e}")
    print("Central2's error shrinks about 4 times each row; Richardson's about 16 times.")

---

## Task 4 — Test the whole family (§8.5)

A test for one class makes an object for a case whose answer you **already
know**, calls it, and compares. A whole family can be tested in one go. A class
is an object too, so `Forward1`, `Backward1`, `Central2` and `Richardson` can go
in a list, and a loop can make an object from each of them.

**On paper first:** for the straight line $f(x) = 3x + 1$, what do Forward1,
Backward1, Central2 and Richardson give at $x = 5$ with $h = 1$? With $h = 7$?
Why does nobody need a calculator for this?

Fill in `test_family()`:

- loop over the list `[Forward1, Backward1, Central2, Richardson]`, and for each
  class make an object for a straight line **whose slope is not 0**, with step
  `h = 1`. Define the line inside `test_family`;
- check its derivative at two or three points with `assert`, and give every
  `assert` a message. `Method.__name__` is the name of the class `Method`, as a
  string, so the message can say which class failed;
- compare with a tolerance, such as `1e-10`, not with `==`.

Why `h = 1`, and not the default `1E-5`? On a straight line every formula is
exact with any step, and with a big step round-off stays out of the way.

A test that passes prints nothing. The TEST cell below runs yours against your
own classes, against stand-ins that are right on every straight line, and then
against four broken ones, one for each member of the family. A good test passes
the first two and catches all four.

In [ ]:
def test_family():
    """Check every class in the family on a straight line, where each one is exact."""
    # TODO: your code here
    return None

In [ ]:
# TEST -- Task 4. Just run it.
# This one is a tool, not an exercise -- you are not expected to read it.
import inspect


def run_with(swaps):
    # Swap in the classes in swaps, run test_family, put the real ones back.
    # Returns the error the test stopped with, or None if it passed.
    real = {name: globals()[name] for name in swaps}
    globals().update(swaps)
    try:
        test_family()
        return None
    except Exception as e:
        return e
    finally:
        globals().update(real)


# Stand-ins that are right on every straight line: lecture's formulas, on top
# of your Diff.
class _Forward(Diff):
    def __call__(self, x):
        f, h = self.f, self.h
        return (f(x+h) - f(x))/h


class _Backward(Diff):
    def __call__(self, x):
        return _Forward(self.f, -self.h)(x)    # Forward1, looking the other way


class _Central(Diff):
    def __call__(self, x):
        f, h = self.f, self.h
        return (f(x+h) - f(x-h))/(2*h)


class _Richardson(Diff):
    def __call__(self, x):
        f, h = self.f, self.h/2               # Central4 from lecture, half the step
        return (4./3)*(f(x+h) - f(x-h))/(2*h) - (1./3)*(f(x+2*h) - f(x-2*h))/(4*h)


class _HalfForward(_Forward):
    def __call__(self, x):
        return super().__call__(x)/2


class _FlippedBackward(_Backward):
    def __call__(self, x):
        return -super().__call__(x)


class _DoubleCentral(_Central):
    def __call__(self, x):
        return 2*super().__call__(x)


class _ShortRichardson(_Richardson):
    def __call__(self, x):
        return 0.75*super().__call__(x)


stand_ins = {"Forward1": _Forward, "Backward1": _Backward,
             "Central2": _Central, "Richardson": _Richardson}
broken = [
    ("3. A Forward1 that gives half the slope:  ", "Forward1", _HalfForward),
    ("4. A Backward1 with the sign flipped:     ", "Backward1", _FlippedBackward),
    ("5. A Central2 that gives twice the slope: ", "Central2", _DoubleCentral),
    ("6. A Richardson that gives 3/4 of it:     ", "Richardson", _ShortRichardson),
]
# So that a message with Method.__name__ names the class it stands in for.
for name, cls in stand_ins.items():
    cls.__name__ = name
for label, name, cls in broken:
    cls.__name__ = name


def cube(x):
    return x**3


try:
    yours_done = (issubclass(Backward1, Diff) and issubclass(Richardson, Central2)
                  and abs(Backward1(cube, 2)(1) - 1) < 1e-9
                  and abs(Richardson(cube, 2)(1) - 3) < 1e-9)
except Exception:
    yours_done = False

if "TODO: your code here" in inspect.getsource(test_family):
    print("Not implemented yet -- test_family does not check anything.")
    print("(Already filled in? Delete the line '# TODO: your code here' and run its cell.)")
else:
    good = True
    if not yours_done:
        print("1. Skipped: your Backward1 or Richardson does not pass its own TEST yet.")
        print("   Get those cells to print PASSED, then run this one again.")
    else:
        error = run_with({})
        if error is None:
            print("1. Your four classes:                     test passed (silently, as it should)")
        else:
            good = False
            print("1. Your four classes:                     test FAILED --", type(error).__name__ + ":", error)
    error = run_with(stand_ins)
    if error is None:
        print("2. Stand-ins, right on every straight line: test passed")
    else:
        good = False
        print("2. Stand-ins, right on every straight line: test FAILED --", type(error).__name__ + ":", error)
        if ".values(" in inspect.getsource(test_family):
            print("   Your test uses values, so Task 1 has to work first.")
        print("   Is the slope your test expects the slope of its line? Is the step h = 1?")
    caught_all = True
    for label, name, fake in broken:
        swaps = dict(stand_ins)
        swaps[name] = fake
        caught = run_with(swaps) is not None
        caught_all = caught_all and caught
        print(label, "CAUGHT" if caught else "missed")
    if not caught_all:
        print("Your test let a broken class through. Does it loop over all four classes,")
        print("and does its line have a slope that is not 0?")
    elif good and yours_done:
        print("PASSED")

If line 1 says `FAILED` with an `AssertionError`, read its message: your test
says one of your classes is wrong. Either that class or the test has a mistake.
Work out a value by hand to decide which.

---

## Daylight, in code

Nothing to write here. The cell below hands the family a real function: the
hours of daylight in Atlanta, from Lab 9. Days are numbered through the year,
and a step of `h = 1` is one day, so `Forward1(atlanta, 1)(day)` is tomorrow's
daylight minus today's, in hours. That is Lab 9's `daily_change`. Times 60, it
is in minutes.

In [ ]:
# Just run this one. It is not graded.
import inspect
import numpy as np


def finished(cls):
    for member in vars(cls).values():
        if inspect.isfunction(member) and "TODO: your code here" in inspect.getsource(member):
            return False
    return True


def declination(day):
    """Lab 9: the declination of the sun, in degrees, on day number day."""
    return 23.44 * np.sin(2 * np.pi * (day + 284) / 365)


def day_length(day, latitude):
    """Lab 9: hours from sunrise to sunset on day number day, at the given latitude."""
    phi = latitude * np.pi / 180
    delta = declination(day) * np.pi / 180
    return 24 / np.pi * np.arccos(-np.tan(phi) * np.tan(delta))


def atlanta(day):
    # The family wants a function of one variable.
    return day_length(day, 33.75)


family = [Forward1, Backward1, Central2, Richardson]
dates = [('Mar 20', 79), ('Jun 21', 172), ('Sep 22', 265), ('Nov 20', 324), ('Dec 21', 355)]

if not (finished(Diff) and finished(Backward1) and finished(Richardson)):
    print("Finish Tasks 1-3, run their cells, then run this one again.")
    print()
    print("Already finished, and every TEST cell printed PASSED? Then the line")
    print("'# TODO: your code here' is probably still in one of your methods.")
    print("Delete it, run that cell, then run this one again.")
elif not (issubclass(Backward1, Diff) and issubclass(Richardson, Central2)):
    print("Backward1 or Richardson was made from an older Diff: the Task 1 cell was run")
    print("again after theirs. Run the Backward1 and Richardson cells again, then this one.")
else:
    today = atlanta(324)
    hours = int(today)
    minutes = round(60 * (today - hours))
    print(f"Today, November 20 (day 324), Atlanta gets {hours} hours {minutes} minutes of daylight.")
    print()
    print("Minutes of daylight gained per day (negative: lost), with a step of one day:")
    print()
    print(" " * 8 + "".join(f"{Method.__name__:>12}" for Method in family))
    days = [day for name, day in dates]
    columns = [Method(atlanta, 1).values(days) for Method in family]
    for i in range(len(dates)):
        print(f"{dates[i][0]:8}" + "".join(f"{60 * column[i]:12.3f}" for column in columns))

- Today the days are getting shorter by about a minute and a quarter a day.
  Forward1 and Backward1 disagree by about two seconds. If you had to report one
  number, which column would you take it from, and why?
- Look at December 21. Forward1 says the days are already getting longer, and
  Backward1 says they are still getting shorter. Can both be right? What does
  Central2 say?
- Lab 9's `daily_change` computed one of these four columns. Which one?

One last thing about the hierarchy. **Predict** what each line prints, then run
the cell.

In [ ]:
# Predict first, then run. Not graded.
r = Richardson(atlanta, 1)
print(isinstance(r, Richardson), isinstance(r, Central2), isinstance(r, Diff))
print(isinstance(r, Forward1), issubclass(Richardson, Diff), issubclass(Diff, Richardson))

plain = Diff(atlanta, 1)
# Remove the # from the next line, predict what happens, and run the cell again.
# print(plain.values([324]))

`values` works for every member of the family, but not for a plain `Diff`. Read
the error: what does `Diff` not have that every one of its subclasses does?

---

## Submitting

### Step 1 — Build `lab13_daylight.py` from your work

The cell below collects your classes and your test function and writes them
into `lab13_daylight.py`, which sits in this same folder. `Forward1` and
`Central2` go in too, as they were given: `Richardson` and your test need them.

**Run every task cell first**, from the top, so that everything exists. Then
run this.

Change your mind about an answer later? Fix it in the task cell, run that cell,
and run this one again: the file is rewritten from scratch each time.

In [ ]:
# Run this to build lab13_daylight.py from the classes and the test you wrote above.
# This one is a tool, not an exercise -- you are not expected to read it.
import ast
import inspect
import json
import linecache
import os

NAMES = ["Diff", "Forward1", "Central2", "Backward1", "Richardson", "test_family"]
GIVEN = ["Forward1", "Central2"]
PLACEHOLDERS = {"Diff": 1, "Backward1": 1, "Richardson": 1, "test_family": 1}
TODO = "TODO: your code here"
OUTFILE = "lab13_daylight.py"
NOTEBOOK = "lab13.ipynb"

HEADER = '''"""
MATH 170 - Lab 13: How Fast Are the Days Getting Shorter?
Fall 2026

Built from lab13.ipynb by the build cell at the end of the notebook.
To change something: edit the task cell, run it, and run the build cell again.
"""

'''


def cut(source, name):
    # The class or function called name, cut out of one cell's code.
    try:
        tree = ast.parse(source)
    except SyntaxError:
        return None
    found = None
    for node in tree.body:
        if isinstance(node, (ast.ClassDef, ast.FunctionDef)) and node.name == name:
            found = ast.get_source_segment(source, node)
    return found


def from_memory(name):
    # The class or function as you last ran it in this notebook.
    thing = globals().get(name)
    if inspect.isfunction(thing):
        try:
            return inspect.getsource(thing)
        except (OSError, TypeError):
            return None
    if inspect.isclass(thing):
        # A notebook cannot hand back a class's code directly, so find the cell
        # through one of its methods and cut the class out of that cell.
        for member in vars(thing).values():
            if inspect.isfunction(member):
                cell = "".join(linecache.getlines(member.__code__.co_filename))
                found = cut(cell, name)
                if found is not None:
                    return found
    return None


def from_notebook(name):
    # Fallback: dig it out of the saved .ipynb file.
    if not os.path.exists(NOTEBOOK):
        return None
    with open(NOTEBOOK) as f:
        cells = json.load(f)["cells"]
    latest = None
    for cell in cells:
        if cell["cell_type"] == "code":
            found = cut("".join(cell["source"]), name)
            if found is not None:
                latest = found
    return latest


pieces = []
report = []
worked_on = 0
for name in NAMES:
    source = from_memory(name)
    if source is None:
        source = from_notebook(name)
    if source is None:
        if name in GIVEN:
            report.append("  MISSING      " + name + " -- it came with the Task 1 cell. Put that cell"
                          " back as it was given and run it")
        else:
            report.append("  MISSING      " + name + " -- write it (spelled exactly like this) and run its cell")
        continue
    pieces.append(source.rstrip() + "\n")
    if name in GIVEN:
        report.append("  given        " + name)
        continue
    left = source.count(TODO)
    if left == 0:
        report.append("  written      " + name)
        worked_on = worked_on + 1
    elif left < PLACEHOLDERS[name]:
        report.append("  partly done  " + name + " -- " + str(left) + " method(s) still have the line"
                      " '# TODO: your code here'")
        worked_on = worked_on + 1
    else:
        report.append("  unfinished   " + name + " -- still the placeholder, or you kept the line"
                      " '# TODO: your code here' (delete it, run that cell, then this one)")

print("\n".join(report))
print()

if worked_on == 0:
    print("Nothing to build yet -- none of the four tasks is filled in.")
    print("Work through the tasks above, run their cells, then run this one again.")
    print("Already filled them in? Delete the line '# TODO: your code here' from each")
    print("method, run its cell, then run this one again.")
else:
    with open(OUTFILE, "w") as f:
        f.write(HEADER + "\n\n".join(pieces))
    print("Wrote " + OUTFILE + " in " + os.getcwd())
    print("Now run the check below.")

### Step 2 — Check the file before you push

In [ ]:
# Run this. It imports YOUR lab13_daylight.py and checks it.
# The try/except is a tool, not an exercise: a crash is printed, not raised.
import importlib
import lab13_daylight
importlib.reload(lab13_daylight)


def show(label, call, expected):
    try:
        print(label, "->", repr(call()), "   should be", expected)
    except Exception as e:
        print(label, "-> crashed with", type(e).__name__ + ":", e)


def square(x):
    return x*x


def cube(x):
    return x**3


show("Central2(square, 1).values([0, 1, 2])",
     lambda: lab13_daylight.Central2(square, 1).values([0, 1, 2]), "[0.0, 2.0, 4.0]")
show("Backward1(cube, 2)(1)", lambda: lab13_daylight.Backward1(cube, 2)(1), "1.0")
show("Richardson(cube, 2)(1)", lambda: lab13_daylight.Richardson(cube, 2)(1), "3.0")
show("test_family()", lambda: lab13_daylight.test_family(), "None -- a passing test returns nothing")
print()
print("None where a number should be, or a crash that mentions NoneType: that method")
print("was not copied over. NameError: something a class uses was made outside it in")
print("the notebook. Move it inside, run its cell, then the build cell.")
print("Anything else -- go on.")

### Step 3 — Push your work to your own repository

From inside your `math170-yourname` folder:

```
git status
git add lab13/lab13_daylight.py
git commit -m "Complete Lab 13"
git push
```

`git status` will also list your notebook, `lab13.ipynb`, as modified. It stays
on your laptop. Stage the one file Gradescope needs, by name.

**Do not use `git add .`** It stages everything the folder happens to contain
that day. Once it is pushed it is in your repository's history, and taking it
back out is far more work than typing one filename.

This sends the file to `origin`, **your** repository. (`upstream` is the course
repository. You pull labs from it; you never push to it.)

Refresh your repository page on GitHub and check that `lab13/lab13_daylight.py`
is there, with your code in it. Gradescope reads your repository, so if the file
is not on GitHub there is nothing for it to grade.

**Pushing does not submit the lab.** It backs your work up. Submission is next.

### Step 4 — Submit on Gradescope

Open **gradescope.com directly in your browser**, **not through Canvas**, or the
GitHub connection will be refused.

Open **MATH 170 → Lab 13** and click **Submit**, then:

1. Choose **GitHub**
2. Pick your repository `math170-yourname` and the `main` branch
3. Submit

Gradescope takes a **snapshot** of your repository at that moment. If you push
more work afterwards, submit again: pushing alone changes nothing on Gradescope.

If GitHub will not connect, upload `lab13_daylight.py` directly instead. That
works just as well.

Wait for the autograder to finish. You should see **100 / 100**. You may
resubmit as many times as you like; the last one counts.

---

## Before you leave

- [ ] All four tasks written, and their TEST cells run
- [ ] Ran the build cell, and the check in Step 2 printed no `None` and no crash
- [ ] Committed and pushed — `lab13/lab13_daylight.py` visible in your repository on GitHub
- [ ] Gradescope shows **100 / 100** for Lab 13
- [ ] You signed the attendance sheet